# Семинар 8 [17-25-2]. Бонусное задание: исследование на симуляции (+5 баллов)

**ФИО:** ...
**Группа:** ...
**Вариант (номер в списке группы):** ...

### Как устроено задание

Пять этапов, каждый — **1 балл**. Код почти готов: в местах `# TODO` нужно дописать одну-три строки. **Баллы ставятся за интерпретацию**: за прогноз, сделанный *до* запуска, за объяснение того, что получилось, и за вывод своими словами.

В каждом этапе есть три текстовые ячейки:

- **Прогноз (до запуска)** — заполняется *до* того, как вы запустите код этапа. Неверный прогноз не снижает балл, если потом вы объяснили, почему он не сбылся.
- **Что получилось** — числа и графики, которые вы увидели.
- **Интерпретация** — почему так получилось, со ссылкой на определения и формулы семинара.

**Защита:** преподаватель может попросить изменить один параметр и до запуска предсказать, как изменится график. Ответ «так сказала нейросеть» баллов не приносит, а умение объяснить свой график — приносит.

Как работать в Colab: загрузите этот файл через **Файл → Загрузить блокнот**. Он сохранится на вашем Google Диске в папке «Colab Notebooks», дальше открывайте его оттуда. Для сдачи скачайте заполненный ноутбук: **Файл → Скачать → Скачать .ipynb**.

Подсказки по `scipy` — в разделе 5 ноутбука семинара. Помните: в `scipy` `cdf(x)` $=P(X\le x)$, а на лекции $F(x)=P(X<x)$.

In [ ]:
VARIANT = 0   # ← впишите номер своего варианта (1–30)

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, integrate

assert 1 <= VARIANT <= 30, "Впишите номер варианта от 1 до 30"
rng = np.random.default_rng(800 + VARIANT)
plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# Параметры вашего варианта
N_HIVES = 4 + VARIANT % 4                               # число испытаний (этап 1)
P_VARROA = round(0.15 + 0.01 * ((VARIANT * 7) % 20), 2)  # вероятность успеха (этап 1)
K = 1 + VARIANT % 3                                     # степень в плотности (этап 2)
L = 4 + (VARIANT // 3) % 3                              # длина отрезка, см (этап 2)
A, B = round(L * 0.25, 2), round(L * 0.6, 2)            # интервал (этап 2)
M_POW = [2, 3, 0.5, 4, 1.5][VARIANT % 5]                # показатель F(x) = (x/L)^m (этап 4)
print(f"Вариант {VARIANT}: испытаний {N_HIVES}, p = {P_VARROA}; плотность c·x^{K}·({L}-x) на [0; {L}], "
      f"интервал ({A}; {B}); m = {M_POW}")

In [ ]:
# Вспомогательные функции (готовы, менять не нужно)
def ecdf(sample, x):
    # Эмпирическая функция распределения: доля наблюдений, строго меньших x
    sample = np.sort(sample)
    return np.searchsorted(sample, x, side="left") / len(sample)

def max_gap(sample, cdf):
    # Наибольшее расхождение эмпирической F_n и теоретической F (проверяем слева и справа от каждой ступеньки)
    s = np.sort(sample)
    n = len(s)
    F = cdf(s)
    return max(np.max(np.arange(1, n + 1) / n - F), np.max(F - np.arange(n) / n))

---
## Этап 1. Ряд распределения и лесенка $F(x)$ (1 балл)

Проводится `N_HIVES` независимых испытаний, в каждом событие наступает с вероятностью `P_VARROA`. $X$ — число наступлений события.

**До запуска** посчитайте руками: $P(X=0)$, $P(X=1)$ и $P(X<2)$. Затем предскажите, какое число получится, если неосторожно написать `stats.binom.cdf(2, n, p)` вместо $P(X<2)$, и больше оно или меньше правильного.

**Прогноз (до запуска):**

- $P(X=0)=$ ...
- $P(X=1)=$ ...
- $P(X<2)=$ ...
- `cdf(2)` даст ... , потому что ...

In [ ]:
apiaries = (rng.random((10_000, N_HIVES)) < P_VARROA).sum(axis=1)   # 10 000 серий испытаний
k = np.arange(N_HIVES + 1)

# TODO: эмпирические частоты каждого k (подсказка: np.bincount(..., minlength=N_HIVES + 1) / число серий)
freq = ...

# TODO: P(X < 2) через scipy (вспомните, что cdf в scipy — это P(X <= x))
p_less_2 = ...

print(pd.DataFrame({"k": k, "частота": np.round(freq, 4),
                    "формула Бернулли": np.round(stats.binom.pmf(k, N_HIVES, P_VARROA), 4)}).set_index("k"))
print("P(X < 2) =", round(p_less_2, 4), " | доля серий с X < 2:", round(np.mean(apiaries < 2), 4),
      " | cdf(2) =", round(stats.binom.cdf(2, N_HIVES, P_VARROA), 4))

xs = np.linspace(-1, N_HIVES + 1, 600)
plt.step(xs, ecdf(apiaries, xs), where="post", label="эмпирическая F_n(x)")
plt.plot(k, stats.binom.cdf(k - 1, N_HIVES, P_VARROA), "ko", label="F(k) = P(X < k)")
plt.xlabel("число наступлений события"); plt.legend(); plt.show()

**Что получилось:**

...

**Интерпретация:**

...

---
## Этап 2. Плотность и площадь (1 балл)

Непрерывная случайная величина $X$ имеет плотность $f(x)=c\,x^K(L-x)$ на $[0;L]$ и $0$ вне отрезка.

**До запуска** найдите руками $c$ из условия нормировки и вероятность $P(A<X<B)$. Предскажите, где у плотности максимум и выше ли он единицы.

**Прогноз (до запуска):**

- $c=$ ...
- $P(A<X<B)=$ ...
- максимум плотности в точке ... , значение ...

In [ ]:
f_raw = lambda x: x**K * (L - x)

# TODO: найдите c численно (integrate.quad возвращает пару (значение, погрешность))
c = ...

f = lambda x: np.where((x >= 0) & (x <= L), c * f_raw(np.asarray(x, dtype=float)), 0.0)
roots = L * stats.beta(K + 1, 2).rvs(size=3000, random_state=rng)   # 3000 значений с такой плотностью

# TODO: P(A < X < B) двумя способами: интегралом от f и долей значений в интервале
p_integral = ...
p_freq = ...

print(f"c = {c:.5f};  P({A} < X < {B}): интеграл {p_integral:.4f}, доля в симуляции {p_freq:.4f}")
xs = np.linspace(0, L, 400)
plt.hist(roots, bins=30, density=True, alpha=0.5, label="3000 значений")
plt.plot(xs, f(xs), "k", lw=2, label="f(x)")
m = (xs > A) & (xs < B)
plt.fill_between(xs[m], f(xs[m]), alpha=0.3, color="C1", label="P(A < X < B)")
plt.xlabel("см"); plt.legend(); plt.show()

**Что получилось:**

...

**Интерпретация:**

...

---
## Этап 3. Оценка параметра по функции распределения (1 балл)

Время безотказной работы прибора (тыс. часов) подчиняется показательному закону $F(t)=1-e^{-\lambda t}$ с неизвестным $\lambda$. Есть данные по 60 приборам вашего варианта (`lifetimes`). Оценка: $\hat\lambda(t_0)=-\ln(\text{доля с } T>t_0)/t_0$.

Выбор $t_0$ свободный. **До запуска** предскажите: при каком $t_0$ из $\{0{,}2;\ 1;\ 3;\ 8\}$ оценка будет самой надёжной (меньше всего «скакать» от выборки к выборке) и почему слишком маленькое и слишком большое $t_0$ плохи.

**Прогноз (до запуска):**

- лучшее $t_0$: ...
- почему: ...

In [ ]:
LAMBDA_TRUE = round(0.25 + 0.025 * (VARIANT % 11), 3)   # не подглядывайте до прогноза :)
lifetimes = stats.expon(scale=1 / LAMBDA_TRUE).rvs(size=60, random_state=rng)

def lam_hat(data, t0):
    # TODO: оценка λ по доле наблюдений, больших t0 (если доля 0 или 1, верните np.nan)
    share = ...
    return ...

T0S = [0.2, 1, 3, 8]
print("Оценки по вашим 60 приборам:", {t0: round(lam_hat(lifetimes, t0), 3) for t0 in T0S})

# Разброс оценок: 2000 повторов эксперимента с 60 приборами
spread = {}
for t0 in T0S:
    est = np.array([lam_hat(s, t0) for s in stats.expon(scale=1 / LAMBDA_TRUE).rvs(size=(2000, 60), random_state=rng)])
    est = est[~np.isnan(est)]
    spread[t0] = np.quantile(est, [0.05, 0.95])
    plt.hist(est, bins=50, range=(0, 3 * LAMBDA_TRUE), density=True, histtype="step", lw=2, label=f"t0 = {t0}")
plt.axvline(LAMBDA_TRUE, color="black", ls="--", label="истинное λ")
plt.xlabel("оценка λ"); plt.legend(); plt.show()
for t0, (lo, hi) in spread.items():
    print(f"t0 = {t0}: 90 % оценок в [{lo:.3f}; {hi:.3f}], ширина {hi - lo:.3f}")
print("Истинное λ =", LAMBDA_TRUE)

**Что получилось:**

...

**Интерпретация:**

...

---
## Этап 4. Своя случайная величина из равномерной (1 балл)

Случайная величина $X$ имеет функцию распределения $F(x)=(x/L)^m$ на $[0;L]$, $m$ = `M_POW`. Компьютер умеет генерировать только равномерное $U$ на $[0;1]$.

**До запуска**: выведите формулу $X=F^{-1}(U)$ и посчитайте, какая доля значений будет больше $L/2$. Как будет выглядеть плотность при вашем $m$: растёт, убывает или постоянна?

**Прогноз (до запуска):**

- $F^{-1}(u)=$ ...
- $P(X>L/2)=$ ...
- форма плотности: ...

In [ ]:
u = rng.random(50_000)

# TODO: преобразуйте u в X по формуле X = F^{-1}(U)
fruits = ...

print("Доля значений больше L/2 в симуляции:", round(np.mean(fruits > L / 2), 4))
xs = np.linspace(0.001, L, 400)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(fruits, bins=50, density=True, alpha=0.5, label="F⁻¹(U)")
axes[0].plot(xs, M_POW / L * (xs / L) ** (M_POW - 1), "k", lw=2, label="f(x) = F'(x)")
axes[0].set_ylim(0, 3 / L * max(M_POW, 1)); axes[0].legend()
axes[1].step(xs, ecdf(fruits, xs), where="post", label="эмпирическая F_n")
axes[1].plot(xs, (xs / L) ** M_POW, "k--", label="F(x)"); axes[1].legend()
plt.tight_layout(); plt.show()

**Что получилось:**

...

**Интерпретация:**

...

---
## Этап 5. Какой это закон? (1 балл)

Имеется выборка из 80 измерений времени ожидания (`plots`, минуты). Есть три гипотезы:

1. равномерный закон на $[0;10]$;
2. показательный закон с $\lambda=0{,}2$;
3. закон из этапа 2, растянутый на $[0;10]$: $f(x)=c\,x^K(10-x)$.

Мерило — наибольшее расхождение $D=\max_x|F_n(x)-F(x)|$ эмпирической и теоретической функций распределения. Но насколько большим $D$ бывает **случайно**, даже если гипотеза верна? Это покажет симуляция: 2000 раз генерируем 80 делянок по самой гипотезе и смотрим, какие $D$ получаются.

**До запуска**: постройте взглядом (или `plt.hist`) гистограмму данных и предположите, какая гипотеза подходит.

**Прогноз (до запуска):**

- гипотеза: ...
- почему: ...

In [ ]:
_true = (VARIANT * 5) % 3
_laws = [stats.uniform(0, 10), stats.expon(scale=5), stats.beta(K + 1, 2, scale=10)]
plots = np.round(_laws[_true].rvs(size=80, random_state=np.random.default_rng(8503 + VARIANT)), 2)
plots = np.clip(plots, 0, None)
print("Данные:", plots)

hypotheses = {"равномерный [0; 10]": _laws[0], "показательный λ = 0,2": _laws[1], "закон этапа 2 на [0; 10]": _laws[2]}
rows = []
for name, law in hypotheses.items():
    D = max_gap(plots, law.cdf)
    # TODO: 2000 раз сгенерируйте 80 делянок из law и посчитайте для каждой выборки max_gap
    D_sim = ...
    rows.append((name, round(D, 3), round(np.quantile(D_sim, 0.95), 3), round(np.mean(D_sim >= D), 3)))
display(pd.DataFrame(rows, columns=["гипотеза", "D по данным", "95 % случайных D меньше", "доля случайных D ≥ наших"]))

xs = np.linspace(0, 20, 400)
plt.step(xs, ecdf(plots, xs), where="post", color="black", lw=2, label="данные")
for name, law in hypotheses.items():
    plt.plot(xs, law.cdf(xs), "--", label=name)
plt.xlabel("сутки"); plt.legend(); plt.show()

**Что получилось:**

...

**Интерпретация:**

...

---
## Итоговый вывод

Три-четыре предложения своими словами: чем функция распределения удобнее плотности при работе с данными и в каком этапе вы в этом убедились.

...